## Installation and Downloads

In [1]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
CUDA available: True
device: cuda
Tesla T4


In [2]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!head -c 300 input.txt

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us

## Creating the Tokenizer

In [3]:
text = open("input.txt", encoding="utf-8").read()
print("length of text:", len(text))

length of text: 1115394


In [4]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("".join(chars))

vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [5]:
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


In [6]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:50])


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [7]:
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(train_data), len(val_data))

1003854 111540


## Batching ( Inputs and Shifted Targets)

In [8]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
print("x:", x)
print("y:", y)

x: tensor([18, 47, 56, 57, 58,  1, 15, 47])
y: tensor([47, 56, 57, 58,  1, 15, 47, 58])


In [9]:
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is {target.item()}")

when input is [18] the target is 47
when input is [18, 47] the target is 56
when input is [18, 47, 56] the target is 57
when input is [18, 47, 56, 57] the target is 58
when input is [18, 47, 56, 57, 58] the target is 1
when input is [18, 47, 56, 57, 58, 1] the target is 15
when input is [18, 47, 56, 57, 58, 1, 15] the target is 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target is 58


In [11]:
torch.manual_seed(1337)

batch_size = 4    # determines the batch size in parallel processing
block_size = 8    # determines the block size or we can say the CONTEXT LENGTH


def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)



xb, yb = get_batch("train")
print("inputs:", xb.shape)
print(xb)
print("targets:", yb.shape)
print(yb)
print("device:", xb.device)

inputs: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
device: cuda:0


## Model

In [14]:
import torch.nn as nn
from torch.nn import functional as F


class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
      super().__init__()
      self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)


  def forward(self, idx, targets = None):
    logits = self.token_embedding_table(idx)
    if targets is None:
      loss = None
    else:
      B,T,C = logits.shape
      loss = F.cross_entropy(logits.view(B*T,C), targets.view(B*T))
    return logits, loss


  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits,_ = self(idx)
      logits = logits[:,-1,:]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples=1)

      idx = torch.cat((idx,idx_next),dim = 1 )
    return idx



### Test Untrained

In [15]:
torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size).to(device)

xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss:", loss.item())

idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

logits shape: torch.Size([4, 8, 65])
loss: 4.705134391784668

pYCXxfRkRZd
wc'wfNfT;OLlTEeC K
jxqPToTb?bXAUG:C-SGJO-33SM:C?YI3a
hs:LVXJFhXeNuwqhObxZ.tSVrddXlaSZaNe


### Train

In [16]:
batch_size = 32
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

print(f"final loss: {loss.item():.4f}")

step 0: loss 4.6684
step 1000: loss 3.6988
step 2000: loss 3.0987
step 3000: loss 2.7456
step 4000: loss 2.5190
step 5000: loss 2.4919
step 6000: loss 2.4829
step 7000: loss 2.4338
step 8000: loss 2.4772
step 9000: loss 2.4017
final loss: 2.4453


In [17]:
print(decode(model.generate(idx, max_new_tokens=300)[0].tolist()))


Wawice my.

HDERarom oroup
Yowh$Frtof isth ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghir.
Tranousel lind me l.
HAule ce hiry:
Supr aisspllw y.
Herindu n Boopetelaves
MP:

Pl, d mothakleo Windo whth eisbyo the m dourive we higend t so mower; te

AN ad nterupt f s ar igr t m:

Thiny a


## Self Attention

In [18]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2          # batch, time, channels
x = torch.randn(B, T, C)
print(x.shape)

# xbow[b, t] = mean of x[b, 0..t]
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]            # (t+1, C)
        xbow[b, t] = xprev.mean(0)

print("x[0]:\n", x[0])
print("xbow[0]:\n", xbow[0])

torch.Size([4, 8, 2])
x[0]:
 tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
xbow[0]:
 tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [19]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print("a:\n", a)
print("b:\n", b)
print("c:\n", c)

a:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b:
 tensor([[8., 6.],
        [5., 2.],
        [4., 4.]])
c:
 tensor([[8.0000, 6.0000],
        [6.5000, 4.0000],
        [5.6667, 4.0000]])


In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x                        # (T,T) @ (B,T,C) -> (B,T,C)
print(x.shape, xbow.shape, xbow2.shape)
print("max difference:", (xbow - xbow2).abs().max().item())
print(torch.allclose(xbow, xbow2, atol=1e-6))

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
print(wei)
xbow3 = wei @ x
print("max difference:", (xbow - xbow3).abs().max().item())
print(torch.allclose(xbow, xbow3, atol=1e-6))

### Self attention head

In [24]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)

wei = q @ k.transpose(-2, -1)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

print("out shape:", out.shape)
print("wei[0]:\n", wei[0])
print("row sums:", wei[0].sum(-1))

out shape: torch.Size([4, 8, 16])
wei[0]:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


In [25]:
torch.manual_seed(1337)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

wei_raw = q @ k.transpose(-2, -1)
wei_scaled = wei_raw * head_size**-0.5

print("variance without scaling:", wei_raw.var().item())
print("variance with scaling:   ", wei_scaled.var().item())

variance without scaling: 18.44363021850586
variance with scaling:    1.1527268886566162


## Multi-Head Attention, Feed-Forward, and the Transformer Block

In [26]:
n_embd = 32       # embedding size (channels)
n_head = 4        # number of attention heads
block_size = 8    # context length
dropout = 0.0     # keep 0 for testing; we'll raise it for real training

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                   # (B, T, hs)
        q = self.query(x)                                 # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)                                 # (B, T, hs)
        return wei @ v                                    # (B, T, hs)

In [27]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, n_embd)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # communicate
        x = x + self.ffwd(self.ln2(x))   # compute
        return x

In [28]:
torch.manual_seed(1337)
x = torch.randn(4, 8, n_embd).to(device)

head = Head(8).to(device)
mha = MultiHeadAttention(n_head, 8).to(device)
ffn = FeedForward().to(device)
block = Block().to(device)

print("Head:", head(x).shape)
print("MultiHead:", mha(x).shape)
print("FeedForward:", ffn(x).shape)
print("Block:", block(x).shape)

print("Block params:", sum(p.numel() for p in block.parameters()))

Head: torch.Size([4, 8, 8])
MultiHead: torch.Size([4, 8, 32])
FeedForward: torch.Size([4, 8, 32])
Block: torch.Size([4, 8, 32])
Block params: 12608


In [29]:
block.eval()
torch.manual_seed(0)
x1 = torch.randn(1, 8, n_embd).to(device)
x2 = x1.clone()
x2[0, 5:] = torch.randn(3, n_embd).to(device)   # alter the "future" positions

with torch.no_grad():
    y1, y2 = block(x1), block(x2)

print("early positions unchanged:", torch.allclose(y1[:, :5], y2[:, :5], atol=1e-5))
print("late positions changed:   ", not torch.allclose(y1[:, 5:], y2[:, 5:], atol=1e-5))

early positions unchanged: True
late positions changed:    True


## GPT

In [30]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_layer):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                          # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T,C)
        x = tok_emb + pos_emb                                              # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                           # (B,T,vocab)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]            # crop to context length
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [31]:
torch.manual_seed(1337)
model = GPT(vocab_size, n_layer=4).to(device)

xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("logits shape:", logits.shape)
print("initial loss:", loss.item())
print("ln(65) =", torch.log(torch.tensor(65.0)).item())
print("params:", sum(p.numel() for p in model.parameters()))

idx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist()))

logits shape: torch.Size([32, 8, 65])
initial loss: 4.175382614135742
ln(65) = 4.174387454986572
params: 54977

?qf;xbDkRZkNdc'wf,ZTkOLlT,ebtK
b:!PjCkMBbzA$3:XaSvgO-33SMBd?gLTa
hX:YVXJthXfNuwqcPMxv.tbaF dXl!DZaLe


In [32]:
n_embd, n_head, block_size, dropout = 384, 6, 256, 0.2

big = GPT(vocab_size, n_layer=6).to(device)
print("big model params: %.2fM" % (sum(p.numel() for p in big.parameters()) / 1e6))

xb_big = torch.randint(0, vocab_size, (64, block_size), device=device)
yb_big = torch.randint(0, vocab_size, (64, block_size), device=device)
_, loss_big = big(xb_big, yb_big)
print("initial loss:", loss_big.item())

if device == "cuda":
    print("GPU memory used (MB):", torch.cuda.max_memory_allocated() // 2**20)

del big  # free memory

big model params: 10.79M
initial loss: 4.245756149291992
GPU memory used (MB): 3819


## Final Training

In [ ]:
import os, time

ckpt_dir = "/content/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

# hyperparameters
batch_size = 64
block_size = 256
n_embd, n_head, n_layer, dropout = 384, 6, 6, 0.2
learning_rate = 3e-4
max_iters = 5000
eval_interval = 500
eval_iters = 100
use_amp = (device == "cuda")

def build():
    """Create a fresh model, optimizer and grad scaler (same seed every time)."""
    torch.manual_seed(1337)
    m = GPT(vocab_size, n_layer=n_layer).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=learning_rate)
    sc = torch.amp.GradScaler("cuda", enabled=use_amp)
    return m, opt, sc

model, optimizer, scaler = build()
print("params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

In [37]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
# Timing test (20 steps). Afterwards the model is rebuilt, so the real run starts fresh.
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
for step in range(20):
    xb, yb = get_batch("train")
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
if device == "cuda":
    torch.cuda.synchronize()
dt = (time.time() - t0) / 20
print(f"{dt:.3f} s/iter  ->  estimated total for {max_iters} iters: {dt*max_iters/60:.1f} min")
if device == "cuda":
    print("peak GPU memory (MB):", torch.cuda.max_memory_allocated() // 2**20)

model, optimizer, scaler = build()      # reset: discard the 20 timing steps
print("model reset, ready for the real run")

In [ ]:
best_val = float("inf")
t0 = time.time()

for step in range(max_iters + 1):

    if step % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}, "
              f"elapsed {(time.time()-t0)/60:.1f} min")
        if losses["val"] < best_val:
            best_val = losses["val"]
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,
                "best_val": best_val,
            }, f"{ckpt_dir}/ckpt.pt")

    if step == max_iters:
        break

    xb, yb = get_batch("train")
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

# final weights (the loop above keeps the best-val checkpoint in ckpt.pt)
torch.save(model.state_dict(), f"{ckpt_dir}/weights_final.pt")
print("training done. best val loss:", round(best_val, 4))

In [ ]:
# Load the BEST checkpoint (lowest val loss) and sample from it
ckpt = torch.load(f"{ckpt_dir}/ckpt.pt", map_location=device)
model.load_state_dict(ckpt["model"])
print("loaded checkpoint from step", ckpt["step"], "val loss", round(ckpt["best_val"], 4))

model.eval()
idx = torch.zeros((1, 1), dtype=torch.long, device=device)
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
    out = model.generate(idx, max_new_tokens=500)
print(decode(out[0].tolist()))

In [ ]:
# Optional: download the best weights to your computer (Colab disk is temporary)
from google.colab import files
torch.save(ckpt["model"], f"{ckpt_dir}/weights_best.pt")
files.download(f"{ckpt_dir}/weights_best.pt")